# Lite R-CNN — Colab T4 성능 측정

Google Colab의 무료 **T4 GPU**에서 Lite R-CNN v2·v3, Faster R-CNN(ResNet50-FPN), MobileNetV3 Faster R-CNN, YOLO26의
**정확도(COCO AP)**와 **속도(지연 시간)**를 같은 조건으로 재고, 추론 설정 **개선 전/후** 결과까지 `results.zip` 하나로 모읍니다.

**처음이면 이것만 하세요**

1. 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU → 저장**
2. 아래 **1. 설정값**은 그대로 둡니다 (브랜치 이름이 다를 때만 고치기).
3. 메뉴 **런타임 → 모두 실행**
   - "Google에서 작성하지 않은 노트북" 경고 → **무시하고 계속하기** (Run anyway)
   - 구글 드라이브 연결 창 → **Google Drive에 연결 → 계정 선택 → 허용** (세션이 끊겨도 결과가 남도록 백업)
   - 두 창 모두 답해야 다음으로 넘어가므로 **처음 1~2분은 자리를 지키세요.** 그 뒤로는 끝날 때까지 손댈 것이 없습니다.
4. 끝나면 `results.zip`이 자동으로 내려받아집니다. 이 파일을 팀에 보내 주세요 (중간에 끊겼다면 맨 아래 **보내 줄 것** 참고).

| 단계 | 하는 일 | T4 예상 시간 |
|---|---|---|
| 0~3 | GPU 확인 · 저장소 받기·설치 · COCO val2017 내려받기 (1 GB) | 5~10분 |
| A | 공정한 기준 비교: Faster R-CNN · MobileNetV3 Faster R-CNN · Lite R-CNN v2 · YOLO26, 이어서 v3 | 25~40분 |
| B | 개선 전/후 비교: 추론 설정 6가지 × 4개 모델 (Faster R-CNN · MobileNet · v2 · v3) | 85~120분 |
| C (선택, 기본 꺼짐) | 팀의 세 모델 평가 스크립트 (+ MobileNet) | 25~35분 |
| 마무리 | `results.zip` 만들기·내려받기 | 1분 |

- 예상 시간은 T4에서 직접 잰 값이 아니라 추정치입니다. 실제 시간은 각 단계 끝에 출력되고 `results.zip`에도 기록됩니다.
- **MobileNetV3 Faster R-CNN**은 팀원 브랜치 `feat/personal-comparison-ci`의 MobileNet 벤치마크와 같은 모델입니다
  (torchvision `fasterrcnn_mobilenet_v3_large_fpn`, COCO_V1 가중치, 320 전용 모델 아님). Lite R-CNN과 같은 MobileNetV3-Large 계열 백본을 쓰는
  기존 경량 Faster R-CNN이라, Lite R-CNN이 그 옆에서 어디쯤인지 보여 줍니다 (BiFPN만의 효과를 가르는 실험은 아님).
  ResNet50 Faster R-CNN보다 훨씬 가벼워 단계마다 몇 분~20분 정도만 더 걸립니다. 빼려면 1번 설정의 `WITH_MOBILENET`을 끕니다.
- **브라우저 탭을 닫거나 컴퓨터가 잠자기에 들어가면 세션이 끊길 수 있습니다.** 실행 중에는 탭을 열어 두세요.
- 각 단계가 끝날 때마다 `results.zip`을 다시 만들어 끝난 단계를 남깁니다. 드라이브 저장을 켰으면 드라이브에도 복사하고,
  오래 걸리는 단계 B는 모델 하나(그 모델의 설정 전부)가 끝날 때마다 복사합니다.
- 처음이라 불안하면 1번 설정에서 `QUICK_TEST`를 켜고 15~25분짜리 점검을 먼저 돌려 보세요 (보고용 결과는 아님).
  점검 결과는 `results_quick.zip`에 따로 담기고, 끄고 다시 돌린 본 측정의 `results.zip`에는 섞이지 않습니다.

자세한 안내(브랜치 올리기, 결과 보기, 문제 해결, 학교 4090 학습): 저장소의 `docs/COLAB.md`

## 0. GPU 확인

아래 표에 **Tesla T4**가 보이면 됩니다. "GPU가 연결되지 않았습니다"가 나오면 메뉴 **런타임 → 런타임 유형 변경**에서
**T4 GPU**를 고르고 저장한 뒤, 이 셀부터 다시 실행하세요.

In [ ]:
# @title 0. GPU 확인
import shutil
import subprocess

import torch

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout)
if not torch.cuda.is_available():
    raise SystemExit("GPU가 연결되지 않았습니다. 메뉴 '런타임 > 런타임 유형 변경'에서 하드웨어 가속기를 "
                     "'T4 GPU'로 바꾸고 저장한 뒤, 이 셀부터 다시 실행하세요.")
DEVICE = "cuda"
GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU: {GPU_NAME} · PyTorch {torch.__version__} · CUDA {torch.version.cuda}")
if "T4" not in GPU_NAME:
    print("[주의] T4가 아닌 GPU입니다. 팀의 지연 시간 목표는 T4 기준이라 이번 숫자는 참고용입니다.")

## 1. 설정값

보통은 **그대로 두면 됩니다.** 바꿨다면 이 셀부터 다시 실행하세요 (또는 다시 **모두 실행**).

| 설정 | 뜻 |
|---|---|
| `REPO_URL`, `BRANCH` | 받을 저장소와 브랜치. 팀 저장소의 `perf/improve` 브랜치 |
| `RUN_LIMIT` | 평가 이미지 수. **0 = COCO val2017 전체 5000장 (보고용)**. 숫자를 넣으면 seed 42로 고른 그만큼만 (세 단계 공통) |
| `QUICK_TEST` | 켜면 이미지 50장과 짧은 지연 시간 측정으로 모든 단계가 도는지만 확인 (15~25분, 보고용 아님) |
| `RUN_STEP_A/B/C` | 실행할 단계. 끝난 단계는 꺼 두고 나머지만 다시 돌릴 수 있습니다 |
| `WITH_MOBILENET` | MobileNet 비교군(MobileNetV3 Faster R-CNN, 팀원 브랜치의 MobileNet 벤치마크)을 넣을지. 끄면 아래 모델 목록에 있어도 세 단계 모두에서 빼고, 단계 C에도 넣지 않습니다 |
| `STEP_A_MODELS` | 단계 A 모델: `faster_rcnn mobilenet_faster_rcnn light_rcnn yolo26` 중에서 (공백으로 구분). `STEP_A_WITH_V3`는 v3를 따로 한 번 더 측정 |
| `STEP_B_MODELS`, `STEP_B_PRESETS` | 단계 B 모델(`faster_rcnn mobilenet v2 v3`)과 추론 설정 (쉼표로 구분, `baseline`은 항상 포함) |
| `SAVE_TO_DRIVE` | 결과를 구글 드라이브 `내 드라이브/Lite-R-CNN-results`에도 저장 (단계가 끝날 때마다, 단계 B는 모델 하나가 끝날 때마다) |

In [ ]:
# @title 1. 설정값 { display-mode: "form" }
REPO_URL = "https://github.com/AI-Engineering-Teamproject/Lite-R-CNN.git"  # @param {type:"string"}
BRANCH = "perf/improve"  # @param {type:"string"}
# @markdown **push 없이 실행** (선택): 왼쪽 파일(폴더 아이콘) 창에 올린 패치 파일 이름. 비워 두면 BRANCH를 받고,
# @markdown BRANCH가 GitHub에 없으면 /content에 올린 .patch 파일 하나를 자동으로 찾아 main에 적용합니다.
PATCH_FILE = ""  # @param {type:"string"}
# @markdown **평가 이미지 수** (0 = val2017 전체 5000장, 보고용)
RUN_LIMIT = 0  # @param {type:"integer"}
# @markdown **빠른 점검** (50장 · 짧은 측정으로 모든 단계가 도는지만 확인, 보고용 아님)
QUICK_TEST = False  # @param {type:"boolean"}
# @markdown **실행할 단계**
RUN_STEP_A = True  # @param {type:"boolean"}
RUN_STEP_B = True  # @param {type:"boolean"}
RUN_STEP_C = False  # @param {type:"boolean"}
# @markdown **MobileNet 비교군** (MobileNetV3 Faster R-CNN, 팀원 브랜치의 MobileNet 벤치마크 · 끄면 세 단계 모두에서 뺌)
WITH_MOBILENET = True  # @param {type:"boolean"}
# @markdown **단계 A**: 모델 (faster_rcnn mobilenet_faster_rcnn light_rcnn yolo26) · v3 추가 측정
STEP_A_MODELS = "faster_rcnn mobilenet_faster_rcnn light_rcnn yolo26"  # @param {type:"string"}
STEP_A_WITH_V3 = True  # @param {type:"boolean"}
# @markdown **단계 B**: 모델 (faster_rcnn mobilenet v2 v3) · 추론 설정
STEP_B_MODELS = "faster_rcnn mobilenet v2 v3"  # @param {type:"string"}
STEP_B_PRESETS = "baseline,deploy,fast,accurate,accurate_fast,gpu_graph"  # @param {type:"string"}
# @markdown **결과를 구글 드라이브에도 저장** (세션이 끊겨도 남음)
SAVE_TO_DRIVE = True  # @param {type:"boolean"}

## 2. 저장소 받기·설치

- `BRANCH` 브랜치를 받고 (`git clone --depth 1`), 필요한 패키지를 설치합니다.
  **Colab에 이미 깔린 CUDA용 PyTorch는 그대로 쓰고 다시 설치하지 않습니다** (버전을 고정해서 설치).
- YOLO26 가중치(`yolo26n.pt`)를 미리 받아 둡니다. Lite R-CNN v2/v3 가중치는 저장소에 들어 있습니다.
- `SAVE_TO_DRIVE`가 켜져 있으면 여기서 구글 드라이브 연결 창이 뜹니다 → **허용**.
- 설치 중 빨간 글씨의 `dependency conflicts` 경고는 Colab에 원래 깔린 다른 패키지에 대한 것이라 무시해도 됩니다.
  마지막에 **준비 완료.**가 나오면 정상입니다.

**받기에 실패하면** ① `BRANCH` 이름이 맞는지, 브랜치를 GitHub에 올렸는지 확인하세요.
② 저장소가 비공개라면 GitHub에서 읽기 권한 토큰을 만들어 왼쪽 **🔑(보안 비밀)**에 이름 `GITHUB_TOKEN`으로 넣고
'노트북 액세스'를 켠 뒤 이 셀을 다시 실행하세요 (방법은 `docs/COLAB.md`).

In [ ]:
# @title 2. 저장소 받기·설치 (처음 한 번 3~5분)
import codecs
import contextlib
import importlib.metadata
import json
import os
import shlex
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd

WORK_DIR = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO_DIR = WORK_DIR / "Lite-R-CNN"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_DIR = REPO_DIR / "outputs" / "colab_logs" / RUN_ID  # 실행마다 새 폴더 (로그는 이어 쓰므로)
STARTED_MARK = REPO_DIR / "outputs" / ".colab_started"  # 이 시각 이후에 생긴 결과만 results.zip에 담음
RESULT_FOLDERS = ("outputs/comparison_runs", "outputs/improvements", "outputs/evaluation_runs",
                  "outputs/reports", "outputs/colab_logs")
STATUS = {}
os.environ["GIT_TERMINAL_PROMPT"] = "0"  # 비공개 저장소에서 비밀번호를 묻느라 멈추지 않게

# QUICK_TEST: 50장 · 짧은 지연 시간 측정 (각 스크립트의 기본 횟수 대신)
LIMIT = RUN_LIMIT if RUN_LIMIT > 0 else (50 if QUICK_TEST else 0)
LIMIT_ARGS = ["--limit", str(LIMIT)] if LIMIT else []
TIMING_A = ["--warmup", "5", "--iterations", "30", "--image-count", "8"] if QUICK_TEST else []
TIMING_B = ["--latency-images", "8", "--warmup", "8", "--iterations", "30"] if QUICK_TEST else []
TIMING_C = (["--timing-images", "8", "--warmup", "8", "--iterations", "30", "--stage-repeats", "1"]
            if QUICK_TEST else [])
MOBILENET_MODELS = {"mobilenet_faster_rcnn", "mobilenet"}  # 단계 A의 모델 이름, 단계 B의 이름


SETUP_OK = False  # 이 셀이 '준비 완료.'까지 끝나면 True (아래 단계 셀들이 확인)

def sh(command, log=None, check=True):
    """명령을 실행하며 출력을 그대로 보여 주고 (log를 주면 파일에도 저장) 성공 여부를 돌려줍니다."""
    shown = command if isinstance(command, str) else shlex.join(str(part) for part in command)
    print(f"$ {shown}", flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1", TQDM_MININTERVAL="2")
    decoder = codecs.getincrementaldecoder("utf-8")("replace")
    with contextlib.ExitStack() as stack:
        log_file = None
        if log is not None:
            Path(log).parent.mkdir(parents=True, exist_ok=True)
            log_file = stack.enter_context(open(log, "a", encoding="utf-8"))
            log_file.write(f"\n$ {shown}\n")
        proc = subprocess.Popen(command, shell=isinstance(command, str), env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        try:
            while chunk := proc.stdout.read1(65536):
                text = decoder.decode(chunk)
                sys.stdout.write(text)
                sys.stdout.flush()
                if log_file:
                    log_file.write(text)
            code = proc.wait()
        except KeyboardInterrupt:  # 중지 버튼: 실행 중인 측정도 멈춤
            proc.terminate()
            proc.wait()
            raise
    if code and check:
        raise SystemExit(f"명령이 실패했습니다 (종료 코드 {code}). 위 출력의 오류 메시지를 확인하세요.")
    return code == 0


def run_step(key, command, log_name):
    """한 단계를 실행하고 결과·걸린 시간을 STATUS에 기록합니다 (실패해도 다음 단계는 진행)."""
    started = time.time()
    ok = sh(command, log=LOG_DIR / log_name, check=False)
    minutes = (time.time() - started) / 60
    STATUS[key] = {"status": "완료" if ok else "실패", "minutes": round(minutes, 1)}
    print(f"\n[{key}] {'완료' if ok else '실패 — 위 출력을 확인하세요'} ({minutes:.0f}분)", flush=True)
    return ok


def chosen_models(text):
    """설정의 모델 목록 (공백으로 구분). WITH_MOBILENET을 끄면 MobileNet 비교군을 뺍니다."""
    return [model for model in text.split() if WITH_MOBILENET or model not in MOBILENET_MODELS]


def safely(label, func, *args):
    """결과 표시는 부가 기능: 실패해도 측정 결과는 그대로 두고 다음 단계로 넘어갑니다."""
    try:
        return func(*args)
    except Exception as exc:  # noqa: BLE001 - 표 하나 때문에 '모두 실행'이 멈추지 않게
        print(f"[주의] {label}을(를) 보여 주지 못했습니다 ({type(exc).__name__}: {exc}). "
              "결과 파일은 그대로 있습니다.")
        return None


def show(obj):
    try:
        from IPython.display import display
    except ImportError:
        print(obj)
    else:
        display(obj)


def show_markdown(path):
    text = Path(path).read_text(encoding="utf-8")
    try:
        from IPython.display import Markdown, display
    except ImportError:
        print(text)
    else:
        display(Markdown(text))


def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:  # noqa: BLE001 - Colab 밖이거나 보안 비밀이 없음
        return None


def find_patch():
    """push 없이 실행할 패치: PATCH_FILE, 아니면 /content에 올린 .patch 파일이 하나뿐일 때 그것."""
    if PATCH_FILE:
        path = Path(PATCH_FILE).expanduser()
        path = path if path.is_absolute() else WORK_DIR / path
        if not path.is_file():
            raise SystemExit(f"패치 파일을 찾지 못했습니다: {path}. 왼쪽 파일 창에 올렸는지, 이름이 맞는지 "
                             "확인하세요.")
        return path
    found = sorted(WORK_DIR.glob("*.patch"))
    return found[0] if len(found) == 1 else None


def clone_with_patch(patch):
    """GitHub의 main을 받고 패치를 적용합니다 (브랜치를 올리지 못했을 때)."""
    print(f"GitHub의 main을 받고 패치 {patch.name}을(를) 적용합니다 (push 없이 실행).")
    if not sh(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
              check=False):
        raise SystemExit("main 브랜치도 받지 못했습니다. 인터넷 연결과 REPO_URL을 확인하세요.")
    if not sh(["git", "-C", str(REPO_DIR), "apply", "--index", str(patch)], check=False):
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        raise SystemExit("패치를 적용하지 못했습니다. 그사이 main이 바뀌었을 수 있으니 새 패치를 받아 "
                         "다시 올리세요.")
    (REPO_DIR / ".git" / "colab_patch").write_text(patch.name, encoding="utf-8")


def clone_repo():
    if (REPO_DIR / ".git" / "colab_patch").is_file():
        print("패치로 만든 저장소가 이미 있어 그대로 씁니다 (측정 결과 파일은 그대로 둡니다).")
        return
    if (REPO_DIR / ".git").is_dir():
        print("이미 받은 저장소가 있어 최신 코드로 갱신합니다 (측정 결과 파일은 그대로 둡니다).")
        fetched = sh(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH],
                     check=False)
        if fetched:
            # "--": 저장소에 FETCH_HEAD라는 파일이 있어도 리비전으로 읽게
            sh(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD", "--"])
        else:
            print("[주의] 갱신하지 못해 이미 받아 둔 코드로 진행합니다.")
        return
    if PATCH_FILE:
        clone_with_patch(find_patch())
        return
    if sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
          check=False):
        return
    token = colab_secret("GITHUB_TOKEN")
    if token and REPO_URL.startswith("https://"):
        print("보안 비밀 GITHUB_TOKEN으로 다시 받습니다.")
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1)
        result = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url,
                                 str(REPO_DIR)], capture_output=True, text=True, check=False)
        print((result.stdout + result.stderr).replace(token, "***"))
        if result.returncode == 0:
            subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL],
                           check=True)
            return
    patch = find_patch()
    if patch is not None:
        clone_with_patch(patch)
        return
    raise SystemExit(f"저장소를 받지 못했습니다. ① BRANCH('{BRANCH}')를 GitHub에 올렸는지 ② 브랜치를 올릴 수 "
                     "없다면 패치 파일을 왼쪽 파일 창에 올리고 이 셀을 다시 실행했는지 ③ 비공개 저장소라면 "
                     "왼쪽 🔑 보안 비밀에 GITHUB_TOKEN을 넣고 노트북 액세스를 켰는지 확인하세요 "
                     "(docs/COLAB.md 참고).")


def install_packages():
    """프로젝트를 설치하되 Colab에 깔린 torch/torchvision(+numpy) 버전은 그대로 고정합니다."""
    pins = []
    for name in ("torch", "torchvision", "numpy"):
        with contextlib.suppress(importlib.metadata.PackageNotFoundError):
            pins.append(f"{name}=={importlib.metadata.version(name)}")
    constraints = WORK_DIR / "colab_constraints.txt"
    command = [sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,compare-yolo]",
               "segno", "plotly>=5.24", "-c", str(constraints)]
    attempts = [pins, [pin for pin in pins if not pin.startswith("numpy==")]]
    for number, attempt in enumerate(attempts, start=1):
        constraints.write_text("\n".join(attempt) + "\n", encoding="utf-8")
        if sh(command, log=LOG_DIR / "install.log", check=False):
            return
        if number < len(attempts):
            print("[주의] 설치가 실패해 numpy 버전 고정만 풀고 한 번 더 시도합니다.")
    raise SystemExit("패키지 설치에 실패했습니다. 위 오류 메시지를 확인하세요.")


def new_result_files():
    """STARTED_MARK 뒤에 새로 생기거나 바뀐 결과 파일 (원래 저장소 파일·다른 방식의 실행 결과 제외)."""
    since = STARTED_MARK.stat().st_mtime
    files = []
    for folder in RESULT_FOLDERS:
        root = REPO_DIR / folder
        if root.is_dir():
            files += [path for path in root.rglob("*")
                      if path.is_file() and path.stat().st_mtime > since]
    return sorted(files)


def write_run_info():
    commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                            capture_output=True, text=True, check=False).stdout.strip()
    info = {
        "run_id": RUN_ID, "written_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "repo_url": REPO_URL, "branch": BRANCH, "commit": commit,
        "gpu": GPU_NAME, "torch": torch.__version__, "cuda": torch.version.cuda,
        "run_limit": RUN_LIMIT, "limit_used": LIMIT, "quick_test": QUICK_TEST,
        "steps": {"A": RUN_STEP_A, "B": RUN_STEP_B, "C": RUN_STEP_C},
        "with_mobilenet": WITH_MOBILENET,
        "step_a_models": STEP_A_MODELS, "step_a_with_v3": STEP_A_WITH_V3,
        "step_b_models": STEP_B_MODELS, "step_b_presets": STEP_B_PRESETS,
        "status": STATUS,
    }
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    (LOG_DIR / f"colab_run_{RUN_ID}.json").write_text(
        json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")


def pack_results():
    """같은 런타임의 결과를 results.zip으로 묶고, 드라이브 저장이 켜져 있으면 복사합니다."""
    write_run_info()
    files = new_result_files()
    zip_path = WORK_DIR / ("results_quick.zip" if QUICK_TEST else "results.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in files:
            archive.write(path, path.relative_to(REPO_DIR).as_posix())
    print(f"{zip_path} ({len(files)}개 파일, {zip_path.stat().st_size / 1e6:.1f} MB)")
    if DRIVE_DIR is not None:
        try:
            target = DRIVE_DIR / f"{zip_path.stem}_{RUN_ID}.zip"
            shutil.copy2(zip_path, target)
            print(f"구글 드라이브에도 저장: {target}")
        except OSError as exc:
            print(f"[주의] 드라이브에 저장하지 못했습니다 ({exc}).")
    return zip_path


def mirror_to_drive(folder, target, stop, interval=60):
    """모델 하나(그 모델의 설정 전부)가 끝날 때마다(summary.csv가 바뀌면) folder를 target에 복사합니다 (VM이 회수돼도 남게)."""
    seen = None
    while True:
        stopped = stop.wait(interval)  # 단계가 끝나면 그래프까지 한 번 더 복사하고 끝냄
        with contextlib.suppress(OSError):  # 첫 측정 전(summary.csv 없음)이거나 드라이브 오류
            mtime = (folder / "summary.csv").stat().st_mtime
            if stopped or mtime != seen:
                shutil.copytree(folder, target, dirs_exist_ok=True)
                seen = mtime
        if stopped:
            return


DRIVE_DIR = None
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DIR = Path("/content/drive/MyDrive/Lite-R-CNN-results")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        print(f"결과를 구글 드라이브에도 저장합니다: {DRIVE_DIR}")
    except Exception as exc:  # noqa: BLE001 - 창을 닫음, 학교 계정 제한, Colab 밖 실행
        print(f"[주의] 구글 드라이브를 연결하지 못해 드라이브 저장은 건너뜁니다 "
              f"({type(exc).__name__}: {exc})")

clone_repo()
os.chdir(REPO_DIR)
# 처음, 또는 빠른 점검 ↔ 본 측정을 바꿨을 때만 새로 셈 (같은 방식이면 끝난 단계를 계속 담음)
MODE = "quick" if QUICK_TEST else "full"
if not STARTED_MARK.exists() or STARTED_MARK.read_text(encoding="utf-8").strip() != MODE:
    STARTED_MARK.parent.mkdir(parents=True, exist_ok=True)
    STARTED_MARK.write_text(MODE, encoding="utf-8")
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                        text=True, check=False).stdout.strip()
print(f"코드: {BRANCH} @ {commit} ({REPO_DIR})")

install_packages()
# 기존 PYTHONPATH(Colab은 /env/python)는 유지하고 이 저장소의 src를 앞에 둡니다.
os.environ["PYTHONPATH"] = os.pathsep.join(
    part for part in (str(REPO_DIR / "src"), os.environ.get("PYTHONPATH")) if part)
os.environ.setdefault("YOLO_CONFIG_DIR", str(REPO_DIR / "outputs" / ".ultralytics"))
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
probe_code = ("import torch, torchvision, light_rcnn; print(torch.__version__, "
              "torchvision.__version__, torch.cuda.is_available(), light_rcnn.__file__)")
probe = subprocess.run([sys.executable, "-c", probe_code], capture_output=True, text=True,
                       check=False)
print(probe.stdout.strip() or probe.stderr)
if probe.returncode or probe.stdout.split()[0] != torch.__version__:
    raise SystemExit("설치 후 PyTorch를 불러오지 못했거나 버전이 바뀌었습니다. 메뉴 '런타임 > 세션 다시 "
                     "시작' 후 처음부터 다시 실행하세요.")

if RUN_STEP_A and "yolo26" in STEP_A_MODELS.split() and not Path("yolo26n.pt").is_file():
    sh([sys.executable, "-c", "from ultralytics import YOLO; YOLO('yolo26n.pt')"],
       log=LOG_DIR / "install.log", check=False)
if shutil.which("nvidia-smi"):
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    (LOG_DIR / "nvidia-smi.txt").write_text(
        subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout,
        encoding="utf-8")
print("\n준비 완료.")
SETUP_OK = True

## 3. COCO val2017 내려받기

공식 서버(images.cocodataset.org)에서 `val2017.zip`(약 0.8 GB)과 주석 ZIP(약 0.25 GB)을 받아
`data/coco/images/val2017`과 `data/coco/annotations/instances_val2017.json`에 풉니다.
**이미 준비되어 있으면 건너뜁니다.** 공식 서버가 안 되면 팀이 쓰는 Hugging Face 사본으로 자동으로 다시 시도합니다.
마지막에 이미지 5000장과 주석이 맞는지 확인합니다.

In [ ]:
# @title 3. COCO val2017 내려받기 (이미 있으면 건너뜀)
if "REPO_DIR" not in globals() or not globals().get("SETUP_OK", True):
    raise SystemExit("2번 셀(저장소 받기·설치)이 '준비 완료.'까지 끝나야 합니다. 2번 셀 출력의 오류를 "
                     "확인하세요. 런타임을 다시 시작했거나 런타임 유형을 바꿨다면 0번 셀부터 다시 실행하세요.")
import urllib.request

from tqdm.auto import tqdm

os.chdir(REPO_DIR)
COCO_DIR = REPO_DIR / "data" / "coco"
VAL_DIR = COCO_DIR / "images" / "val2017"
ANN_FILE = COCO_DIR / "annotations" / "instances_val2017.json"
ZIP_DIR = WORK_DIR / "coco_zips"
EXPECTED_IMAGES = 5000
COCO_ZIPS = {
    "val2017.zip": "images.cocodataset.org/zips/val2017.zip",
    "annotations_trainval2017.zip":
        "images.cocodataset.org/annotations/annotations_trainval2017.zip",
}


def download(name, address):
    target = ZIP_DIR / name
    if target.is_file() and zipfile.is_zipfile(target):
        print(f"{name}: 이미 받아 둠")
        return target
    ZIP_DIR.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + ".part")
    error = None
    for scheme in ("https", "http"):
        url = f"{scheme}://{address}"
        try:
            with urllib.request.urlopen(url, timeout=60) as response, open(partial, "wb") as out:
                total = int(response.headers.get("Content-Length") or 0)
                with tqdm(total=total or None, unit="B", unit_scale=True, desc=name) as bar:
                    while chunk := response.read(1 << 20):
                        out.write(chunk)
                        bar.update(len(chunk))
            if total and partial.stat().st_size != total:
                raise OSError("다운로드가 중간에 끊겼습니다")
            if not zipfile.is_zipfile(partial):
                raise OSError("ZIP 파일이 아닙니다")
            partial.replace(target)
            return target
        except Exception as exc:  # noqa: BLE001 - https가 안 되면 http로 한 번 더
            error = exc
            print(f"{url}: {type(exc).__name__}: {exc}")
    raise OSError(f"{name}을(를) 받지 못했습니다 ({error})")


def extract(zip_path, wanted, destination):
    """ZIP에서 wanted(name)가 참인 항목만 destination 아래에 풉니다 (경로 조작 항목은 무시)."""
    with zipfile.ZipFile(zip_path) as archive:
        members = [m for m in archive.infolist() if not m.is_dir() and wanted(m.filename)
                   and ".." not in Path(m.filename).parts and not m.filename.startswith("/")]
        for member in tqdm(members, desc=f"압축 풀기: {zip_path.name}", unit="개"):
            target = destination / member.filename
            if target.is_file() and target.stat().st_size == member.file_size:
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, open(target, "wb") as out:
                shutil.copyfileobj(source, out, 1 << 20)
    return len(members)


def coco_ready():
    return (ANN_FILE.is_file() and VAL_DIR.is_dir()
            and sum(1 for _ in VAL_DIR.glob("*.jpg")) >= EXPECTED_IMAGES)


if coco_ready():
    print(f"COCO val2017이 이미 준비되어 있습니다: {COCO_DIR}")
else:
    try:
        val_zip = download("val2017.zip", COCO_ZIPS["val2017.zip"])
        ann_zip = download("annotations_trainval2017.zip",
                           COCO_ZIPS["annotations_trainval2017.zip"])
        extract(val_zip, lambda name: name.startswith("val2017/") and name.endswith(".jpg"),
                COCO_DIR / "images")
        extract(ann_zip, lambda name: name == "annotations/instances_val2017.json", COCO_DIR)
    except Exception as exc:  # noqa: BLE001 - 어떤 이유든 사본으로 다시 시도
        print(f"[주의] 공식 서버에서 준비하지 못했습니다 ({exc}). "
              "Hugging Face 사본으로 다시 시도합니다.")
        sh([sys.executable, "scripts/prepare_hf_coco.py", "--splits", "annotations", "val",
            "--cache-dir", str(WORK_DIR / "hf_cache")], log=LOG_DIR / "coco_download.log")

annotations = json.loads(ANN_FILE.read_text(encoding="utf-8"))
images_on_disk = sum(1 for _ in VAL_DIR.glob("*.jpg"))
missing = [image["file_name"] for image in annotations["images"]
           if not (VAL_DIR / image["file_name"]).is_file()]
print(f"이미지 파일 {images_on_disk}장 · 주석: 이미지 {len(annotations['images'])}장, "
      f"객체 {len(annotations['annotations'])}개, 클래스 {len(annotations['categories'])}개")
if len(annotations["images"]) != EXPECTED_IMAGES or missing:
    raise SystemExit(f"COCO val2017이 완전하지 않습니다 (주석 이미지 "
                     f"{len(annotations['images'])}장, 없는 파일 {len(missing)}개). "
                     "이 셀을 다시 실행하세요.")
del annotations
print("COCO val2017 확인 완료.")

## 단계 A — 공정한 기준 비교 (`scripts/run_comparison.py`)

네 모델(Faster R-CNN ResNet50-FPN, MobileNetV3 Faster R-CNN, Lite R-CNN, YOLO26n)을 **같은 규칙**으로 잽니다
(`docs/MEASUREMENT.md`의 비교 프로토콜). 두 Faster R-CNN은 torchvision COCO_V1 가중치를 받아 쓰고, 다른 모델처럼 640으로 리사이즈합니다
(MobileNet의 torchvision 공식 AP 32.8은 800px 기준이라 여기서는 더 낮게 나올 수 있음).

- **정확도**: COCO val2017 (`RUN_LIMIT`장), 1장씩, pycocotools, 검출 점수 하한 0.001, 이미지당 최대 100개
- **지연 시간**: 디코드된 이미지 → 최종 검출 (전처리·후처리/NMS 포함, CUDA 동기화), 예열 50회 · 반복 500회 · 이미지 64장
- **단계별 시간**: R-CNN 계열의 백본(본체 · FPN/BiFPN) · RPN · RoI 헤드 · 후처리
- 실행 순서: Faster R-CNN → MobileNetV3 Faster R-CNN → Lite R-CNN **v2** → YOLO26n, 그다음 **v3**를 따로 한 번 더
  (v2와 v3는 결과 파일 이름이 같아 한 폴더에 둘 수 없어서입니다)

결과: `outputs/comparison_runs/<실행 시각>/` (v2 실행, v3 실행 각각 한 폴더). 예상 시간: **25~40분** (MobileNet 몫 약 5분).

In [ ]:
# @title 단계 A 실행: 공정한 기준 비교
if "REPO_DIR" not in globals() or not globals().get("SETUP_OK", True):
    raise SystemExit("2번 셀(저장소 받기·설치)이 '준비 완료.'까지 끝나야 합니다. 2번 셀 출력의 오류를 "
                     "확인하세요. 런타임을 다시 시작했거나 런타임 유형을 바꿨다면 0번 셀부터 다시 실행하세요.")
from light_rcnn import viz
from light_rcnn.reporting import (
    comparability_warnings,
    load_reports,
    load_targets,
    select_reports,
)

os.chdir(REPO_DIR)


def summarize_run(folder, tag, table):
    """실행 폴더의 보고서를 표 행으로 모으고 비교 전 확인할 점을 출력합니다 (v2 실행은 그래프도)."""
    reports, errors = load_reports(folder)
    for error in errors:
        print(f"[읽기 실패] {error}")
    selected = select_reports(reports)
    for row in viz.summary_rows(selected):
        if row["모델"] == "Light R-CNN":
            row["모델"] = f"Light R-CNN {tag}"
        table.append(row)
    for warning in comparability_warnings(selected):
        if warning.level != "info":
            print(f"[{viz.LEVEL_LABELS.get(warning.level, warning.level)}] {warning.message}")
    if tag == "v2":
        return viz.tradeoff_figure(selected, load_targets("configs/coco.yaml"), "light")
    return None


def show_table(table):
    columns = ["모델", "AP@[.50:.95] (%)", "AP50 (%)", "AP75 (%)", "p50 (ms)", "p95 (ms)",
               "FPS", "파라미터 (M)", "GFLOPs", "평가 이미지", "반복 횟수", "장치"]
    show(pd.DataFrame(table).reindex(columns=columns).round(2))


if RUN_STEP_A:
    runs_root = REPO_DIR / "outputs" / "comparison_runs"
    plans = [("v2", chosen_models(STEP_A_MODELS), "outputs/coco_4090_v2/best.pth")]
    if STEP_A_WITH_V3:
        plans.append(("v3", ["light_rcnn"], "outputs/coco_4090_v3/best.pth"))
    table, tradeoff = [], None
    for tag, models, weights in plans:
        if not models:
            continue
        before = {path for path in runs_root.glob("*") if path.is_dir()}
        command = [sys.executable, "scripts/run_comparison.py", "--config", "configs/coco.yaml",
                   "--device", DEVICE, "--models", *models, "--light-weights", weights,
                   "--no-dashboard", *LIMIT_ARGS, *TIMING_A]
        if "yolo26" in models:
            command += ["--yolo-weights", "yolo26n.pt"]
        print(f"\n===== 단계 A ({tag}): {' '.join(models)} =====", flush=True)
        run_step(f"A ({tag})", command, f"step_a_{tag}.log")
        created = sorted({path for path in runs_root.glob("*") if path.is_dir()} - before)
        if not created:
            continue
        folder = created[-1]
        STATUS[f"A ({tag})"]["folder"] = folder.relative_to(REPO_DIR).as_posix()
        figure = safely(f"단계 A ({tag}) 결과 표", summarize_run, folder, tag, table)
        tradeoff = figure if figure is not None else tradeoff
    if table:
        safely("단계 A 결과 표", show_table, table)
    if tradeoff is not None and not viz.is_empty(tradeoff):
        safely("정확도 대 속도 그래프", tradeoff.show)
    pack_results()
else:
    STATUS["A"] = {"status": "건너뜀"}
    print("단계 A는 건너뜁니다 (RUN_STEP_A = False).")

## 단계 B — 개선 전/후 비교 (`scripts/compare_improvements.py`)

모델마다 **가중치를 새로 불러와 추론 설정만 바꿔서** (재학습 없음) 같은 COCO 이미지의 AP와 같은 방식의 지연 시간을 잽니다.
공정하도록 **두 기준 모델(Faster R-CNN, MobileNetV3 Faster R-CNN)에도 똑같은 설정을 적용**합니다.

| 설정 | 바뀌는 것 | CPU 스모크에서 본 효과 (참고용) | T4에서 확인할 것 |
|---|---|---|---|
| `baseline` | 모델 기본 설정 (개선 전) | 기준 | 기준 |
| `deploy` | Conv+BN 융합 (channels_last는 CPU에서만 자동 적용) | AP 그대로, 지연 `baseline` 대비 v2 −35~−42% / v3 −34~−38% | GPU에서 융합만으로 얼마나 빨라지는지 |
| `fast` | `deploy` + RPN NMS 전 후보 1000→500 + RPN 점수 하한 0.05 | AP v2 +0.02 / v3 +0.10%p, 지연 `deploy` 대비 v2 −25~−28% / v3 −35~−36% | GPU에서도 RPN 단축이 보이는지 (`deploy` 대비로 판단) |
| `accurate` | `deploy` + 640×640 고정 캔버스(RoI 스케일 고정) + Soft-NMS | AP v2 +3.45 / v3 +1.75%p, 지연 `deploy` 대비 v2 +1~+8% / v3 +2~+3% (v3는 오차 범위) | val 5000장에서도 AP가 오르는지 (기대 +1~2%p, 검증 전) |
| `accurate_fast` | `accurate`에서 Soft-NMS 대신 Matrix NMS | AP v2 +2.11 / v3 +1.25%p (`accurate`보다 낮음), 지연은 `accurate`와 뚜렷한 차이 없음 | GPU에서 `accurate`보다 빠른지 (Soft-NMS가 느릴 때의 대안) |
| `gpu_graph` | Conv+BN 융합 + 고정 캔버스 + 백본 CUDA 그래프 (CUDA 전용) | CPU에서는 측정 불가 | 처음 GPU에서 검증. 실행 오버헤드가 줄어드는지 |

"CPU 스모크"는 4코어 CPU · COCO 128장(학습 이미지라 AP가 부풀려짐)에서 같은 측정을 두 번 한 범위라 방향만 참고하세요.

- 정확도: 1장씩, pycocotools, 검출 점수 하한 0.05 (모델 기본값 = 학습 검증·팀 세 모델 평가 스크립트와 같은 하한, 개선 전/후 비교용).
  **단계 A(하한 0.001)와 AP를 섞어 비교하지 마세요.** `docs/MEASUREMENT.md` 비교 프로토콜(0.001) 숫자가 필요하면 단계 A 결과를 쓰세요.
- 지연 시간: 평가 이미지 중 seed 42로 고른 64장, 디코드된 이미지 → 최종 검출 (CUDA 동기화). **교차 측정**으로 잽니다:
  모델마다 설정을 모두 GPU에 올려 두고 번갈아 예열(설정마다 64회)한 뒤, 라운드마다 설정 순서를 한 칸씩 바꿔 가며
  64장씩 번갈아 잽니다. 라운드 수는 설정 수의 배수라 설정마다 순서의 모든 자리를 같은 횟수만큼 돕니다
  (`STEP_B_PRESETS` 기본값인 설정 6개면 12라운드, 설정마다 768회). 측정 중 GPU·CPU 상태가 바뀌어도 모든 설정에 고르게 섞입니다.
- `summary.md` 표의 `Δp50 (%) [95% CI]`는 `baseline` 대비, `Δp50 vs deploy (%) [95% CI]`는 `deploy` 대비 p50 변화율입니다.
  [ ] 안은 라운드를 짝지어 다시 뽑아 구한 95% 신뢰구간이고, **0을 포함하면 변화 없음**으로 봅니다.
  이 구간은 **이번 실행 안의 흔들림만** 담습니다 (CPU에서 같은 측정을 두 번 했을 때 `baseline` 대비 값은 구간보다 더 움직였음).
  10% 안팎의 차이로 무언가를 정해야 하면 단계 B를 한 번 더 돌려 같은 결과가 나오는지 확인합니다.
- 결과: `outputs/improvements/colab_<실행 시각>/` — `summary.md`(표), `summary.csv`, `figures/`(그래프), `reports/v2`·`reports/v3`(대시보드용,
  두 기준 모델의 보고서는 버전 폴더마다 함께 들어 있음).
  드라이브 저장을 켰으면 모델 하나(그 모델의 설정 전부)가 끝날 때마다 `내 드라이브/Lite-R-CNN-results/step_b_<실행 시각>/`에도 복사합니다 (세션이 끊겨도 끝난 모델은 남음).
- 진행 표시: 모델마다 `[15/24] Light R-CNN v2 · 빠른 설정 (fast)`처럼 설정별 정확도를 먼저 잰 뒤, `[지연 시간] …` 줄과 함께 그 모델의 지연 시간을 한꺼번에 잽니다.
- 한 설정이 실패해도 나머지는 계속 잽니다. 실패 이유는 `summary.md`의 "건너뛰거나 실패한 측정"에 남습니다.

예상 시간: **85~120분** (Faster R-CNN이 절반 가까이, MobileNet은 15~20분).

In [ ]:
# @title 단계 B 실행: 개선 전/후 비교
if "REPO_DIR" not in globals() or not globals().get("SETUP_OK", True):
    raise SystemExit("2번 셀(저장소 받기·설치)이 '준비 완료.'까지 끝나야 합니다. 2번 셀 출력의 오류를 "
                     "확인하세요. 런타임을 다시 시작했거나 런타임 유형을 바꿨다면 0번 셀부터 다시 실행하세요.")
from light_rcnn.inference_opts import PRESET_LABELS

os.chdir(REPO_DIR)
GROUP_LABELS = {"v2": "Light R-CNN v2", "v3": "Light R-CNN v3", "faster_rcnn": "Faster R-CNN",
                "mobilenet": "MobileNetV3 Faster R-CNN",
                "baselines": "Faster R-CNN · MobileNetV3 Faster R-CNN"}


def preset_names(text):
    names = ["baseline"]
    for raw in text.split(","):
        name = raw.strip().lower().replace("-", "_")
        if name and name not in names:
            names.append(name)
    return names


def show_improvements(out_dir, presets):
    """보고서 폴더(reports/v2, reports/v3)마다 설정별 개선 전/후 그래프."""
    for group_dir in sorted(path for path in (out_dir / "reports").glob("*") if path.is_dir()):
        reports, _ = load_reports(group_dir)
        baseline = select_reports(reports, variant="baseline")
        for preset in presets[1:]:
            selected = select_reports(reports, variant=preset)
            if not baseline or not selected:
                continue
            label = PRESET_LABELS.get(preset, preset)
            fig = viz.improvement_figure(baseline, selected, "light", label,
                                         ap_metric="AP_50_95", percentile="p50")
            if viz.is_empty(fig):
                continue
            group = GROUP_LABELS.get(group_dir.name, group_dir.name)
            fig.update_layout(title_text=f"<b>개선 전/후 · {group} · {label}</b>")
            fig.show()


if RUN_STEP_B:
    presets = preset_names(STEP_B_PRESETS)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")  # 다시 실행해도 새 폴더
    out_dir = REPO_DIR / "outputs" / "improvements" / f"colab_{stamp}"
    command = [sys.executable, "scripts/compare_improvements.py", "--config", "configs/coco.yaml",
               "--device", DEVICE, "--models", *chosen_models(STEP_B_MODELS),
               "--presets", ",".join(presets), "--output-dir", out_dir.relative_to(REPO_DIR),
               *LIMIT_ARGS, *TIMING_B]
    stop = threading.Event()
    if DRIVE_DIR is not None:  # 오래 걸리는 단계라 모델 하나가 끝날 때마다 드라이브에도 복사
        mirror_dir = DRIVE_DIR / f"step_b_{'quick_' if QUICK_TEST else ''}{stamp}"
        threading.Thread(target=mirror_to_drive, args=(out_dir, mirror_dir, stop),
                         daemon=True).start()
        print(f"모델 하나의 측정이 끝날 때마다 드라이브에도 복사합니다: {mirror_dir}", flush=True)
    try:
        ok = run_step("B", command, "step_b.log")
    finally:  # 중지 버튼을 눌러도 복사를 멈춤 (마지막으로 한 번 더 복사한 뒤)
        stop.set()
    STATUS["B"]["folder"] = out_dir.relative_to(REPO_DIR).as_posix()
    summary = out_dir / "summary.md"
    if summary.is_file():
        if not ok:
            STATUS["B"]["status"] = "일부 실패 (summary.md의 '건너뛰거나 실패한 측정' 참고)"
        safely("summary.md", show_markdown, summary)
        safely("개선 전/후 그래프", show_improvements, out_dir, presets)
    pack_results()
else:
    STATUS["B"] = {"status": "건너뜀"}
    print("단계 B는 건너뜁니다 (RUN_STEP_B = False).")

## 단계 C (선택) — 팀의 세 모델 평가 스크립트 (`scripts/evaluate_three_models_gpu_full.py`)

팀이 쓰던 세 모델 평가표(`REPORT.md`, `summary.csv`)를 고친 버전으로 다시 만듭니다. 세 모델 모두 RPN 제안 300개 ·
Conv+BN 융합으로 맞추고, 제안 재현율(원본 좌표), 단계별 시간, 제안 300/100개 지연 시간을 함께 잽니다.
`WITH_MOBILENET`이 켜져 있으면 `--with-mobilenet`으로 MobileNetV3 Faster R-CNN을 네 번째 모델(표의 행 하나 추가, 열은 그대로)로
같은 조건에서 함께 잽니다.
단계 A·B와 겹치는 부분이 많아 **기본은 꺼져 있습니다** (`RUN_STEP_C`). 시간이 남으면 켜고 이 셀만 다시 실행하세요.

결과: `outputs/evaluation_runs/colab_<실행 시각>/`. 예상 시간: **25~35분**.

In [ ]:
# @title 단계 C 실행 (선택): 팀 세 모델 평가
if "REPO_DIR" not in globals() or not globals().get("SETUP_OK", True):
    raise SystemExit("2번 셀(저장소 받기·설치)이 '준비 완료.'까지 끝나야 합니다. 2번 셀 출력의 오류를 "
                     "확인하세요. 런타임을 다시 시작했거나 런타임 유형을 바꿨다면 0번 셀부터 다시 실행하세요.")
os.chdir(REPO_DIR)
if RUN_STEP_C:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")  # 다시 실행해도 새 폴더
    out_dir = REPO_DIR / "outputs" / "evaluation_runs" / f"colab_{stamp}"
    command = [sys.executable, "scripts/evaluate_three_models_gpu_full.py", "--device", DEVICE,
               "--output-dir", out_dir.relative_to(REPO_DIR),
               *(["--images", str(LIMIT)] if LIMIT else []),
               *(["--with-mobilenet"] if WITH_MOBILENET else []), *TIMING_C]
    run_step("C", command, "step_c.log")
    STATUS["C"]["folder"] = out_dir.relative_to(REPO_DIR).as_posix()
    if (out_dir / "REPORT.md").is_file():
        safely("REPORT.md", show_markdown, out_dir / "REPORT.md")
    pack_results()
else:
    STATUS["C"] = {"status": "건너뜀"}
    print("단계 C는 건너뜁니다 (RUN_STEP_C = False).")

## 마무리 — 결과 내려받기·보내기

아래 셀이 결과를 `results.zip`으로 묶어 내려받습니다 (빠른 점검이면 `results_quick.zip`). 같은 런타임에서 먼저 끝낸 단계도
함께 담고, 빠른 점검 ↔ 본 측정을 바꾸면 그때부터 새로 담습니다 (빠른 점검 결과가 섞이지 않게).
들어 있는 것: 단계 A `outputs/comparison_runs/…`, 단계 B `outputs/improvements/…`, 단계 C `outputs/evaluation_runs/…`,
실행 기록 `outputs/colab_logs/<실행 시각>/` (단계별 로그, GPU 정보, 설정값과 걸린 시간).

**보내 줄 것**

1. `results.zip` (자동으로 안 받아지면 왼쪽 📁 아이콘 → `results.zip` 오른쪽 클릭 → 다운로드.
   드라이브 저장을 켰다면 `내 드라이브/Lite-R-CNN-results/results_<실행 시각>.zip`도 같은 파일입니다)
   - **중간에 세션이 끊겨 다시 돌렸다면** 새 `results.zip`에는 끊기기 전에 끝난 결과가 없습니다.
     `내 드라이브/Lite-R-CNN-results`의 `results_<시각>.zip`을 **모두**, `step_b_<시각>` 폴더가 있으면 그것도 보내 주세요
     (폴더는 드라이브에서 오른쪽 클릭 → 다운로드하면 zip으로 받아집니다. 이름에 `quick`이 들어간 것은 빠른 점검이라 빼도 됩니다).
2. 오류가 났다면 메뉴 **파일 → 다운로드 → .ipynb 다운로드**로 받은 이 노트북 (출력이 함께 저장되어 원인을 볼 수 있습니다)
3. 한 줄 메모: 어떤 단계를 돌렸는지, 중간에 끊겼는지

결과를 읽는 법과 다음 계획은 저장소의 `docs/PERFORMANCE_PLAN.md`에, 받은 결과를 내 PC의 대시보드로 보는 법은
`docs/COLAB.md`의 '결과 보기'에 있습니다.

**자주 생기는 문제**

- *세션이 끊김*: 다시 연결하면 새 컴퓨터가 배정되어 파일이 사라집니다. 드라이브에는 끝난 단계가 담긴 `results_<시각>.zip`과,
  단계 B 도중이었다면 끝난 모델까지 복사된 `step_b_<시각>` 폴더가 남아 있습니다.
  끝난 단계는 1번 설정에서 끄고 다시 **모두 실행**하세요 (설치·다운로드는 다시 함). 단계 B는 이어서 잴 수 없어 처음부터 다시 잽니다.
  `step_b_<시각>/summary.md`에 들어 있는 모델은 모든 설정이 끝난 것이므로 `STEP_B_MODELS`에서 빼도 됩니다.
  보낼 때는 위 **보내 줄 것** 1번처럼 드라이브의 zip과 폴더를 모두 보내 주세요.
- *GPU 사용 한도에 걸림*: 무료 Colab은 하루 GPU 시간이 정해져 있습니다. 끝난 단계만 보내 주고 나머지는 다음 날 돌려도 됩니다.
- *CUDA out of memory*: 메뉴 **런타임 → 세션 다시 시작** 후 처음부터 다시 실행하세요.
- *단계 B에서 `gpu_graph`만 실패*: 이 설정은 GPU에서 처음 검증하는 것이라 실패할 수 있습니다. 결과를 그대로 보내 주세요.
- *MobileNet 가중치를 받지 못함* (`MobileNetV3 Faster R-CNN` 단계에서 `URLError` 등): 단계 A·C는 그 자리에서 멈추고(앞서 끝난 모델의
  결과는 남음), 단계 B는 MobileNet만 건너뛰고 계속합니다. 1번 설정에서 `WITH_MOBILENET`을 끄고 멈춘 단계만 다시 돌린 뒤 결과를 보내 주세요.

In [ ]:
# @title 마무리: results.zip 만들고 내려받기
if "REPO_DIR" not in globals() or not globals().get("SETUP_OK", True):
    raise SystemExit("2번 셀(저장소 받기·설치)이 '준비 완료.'까지 끝나야 합니다. 2번 셀 출력의 오류를 "
                     "확인하세요. 런타임을 다시 시작했거나 런타임 유형을 바꿨다면 0번 셀부터 다시 실행하세요.")
zip_path = pack_results()
rows = [{"단계": key, "결과": value.get("status"), "걸린 시간 (분)": value.get("minutes"),
         "결과 폴더": value.get("folder")} for key, value in STATUS.items()]
if rows:
    show(pd.DataFrame(rows))
try:
    from google.colab import files
except ImportError:
    print(f"Colab 밖에서 실행 중입니다. 결과 파일: {zip_path}")
else:
    try:
        files.download(str(zip_path))
        print("브라우저 내려받기를 시작했습니다. 안 되면 왼쪽 📁 아이콘에서 "
              f"{zip_path.name}을(를) 오른쪽 클릭 → 다운로드하세요.")
    except Exception as exc:  # noqa: BLE001 - 탭이 닫혔거나 브라우저가 막음
        print(f"[주의] 자동 내려받기 실패 ({exc}). 왼쪽 📁 아이콘에서 {zip_path.name}을(를) "
              "오른쪽 클릭 → 다운로드하세요.")